# Neural population forecasting

Formal six-session, three-seed experiment using the manuscript objective and transition kernel.
The notebook and command line share all configuration, checkpoints, and paper exports.
Restart the kernel after applying the patch, then run all cells.


In [ ]:
from pathlib import Path
import sys, json
import torch
import pandas as pd
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/neural_forecasting_v2.json").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the count-flow-map project.")
sys.path.insert(0, str(ROOT))
from countflow.neural_experiment import read_config, download_data, run_experiment

CONFIG_PATH = ROOT / "configs/neural_forecasting_v2.json"
NEURAL_CONFIG = read_config(CONFIG_PATH)
if NEURAL_CONFIG["purpose"] != "paper":
    raise ValueError("This notebook displays only the formal paper experiment.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
OUTPUT = ROOT / NEURAL_CONFIG["output_dir"]
print("Device:", DEVICE)
print("Output and checkpoints:", OUTPUT)
print(json.dumps(NEURAL_CONFIG, indent=2))
STAGE = "all"  # all, train, evaluate, report

## Data

Download the six configured sessions once. Already present files are reused. Download separately on a node with internet access if needed: `python scripts/run_neural_forecasting.py --download-only`.

In [ ]:
DOWNLOAD_IF_MISSING = True
DATA = ROOT / NEURAL_CONFIG["data_root"]
REQUIRED = [DATA / "metadata.json"] + [DATA / f"session_{s:03d}.npy" for s in NEURAL_CONFIG["sessions"]]
missing = [p for p in REQUIRED if not p.exists()]
if STAGE == "report":
    print("Redrawing saved results; data download is unnecessary.")
elif missing and DOWNLOAD_IF_MISSING:
    download_data(NEURAL_CONFIG, ROOT)
elif missing:
    raise FileNotFoundError(f"Missing {len(missing)} data files. Run the downloader first.")
else:
    print("All configured data files are present.")

## Formal run

`all` trains or resumes the formal candidates, evaluates the selected models, and exports reports under the existing `outputs/neural_paper_v2` directory. Previous Count Flow Map and Count-FM fits, evaluations, and reports are moved to `count_models_before_original_method/`. Both count models start from fresh initialization. Compatible direct-mixture, Transformer-NB, and GLM results are reused. Subsequent interruptions resume the new fits from `last.pt`.

For the neural application we use exactly
$$
\mathcal L=\mathcal L_{\mathrm{diag}}+\alpha\mathcal L_{\mathrm{CK}}.
$$
There is no supervised endpoint loss and no source-direction mask. The bridge, count support, mixture family, and samplers are unchanged from the manuscript construction. Finite-time corrections use $\Delta=t-s$ inside the exponential, as specified in its kernel parameterization.

For CK training, $Z\sim K_{\theta,s,u}(\cdot\mid X_s,H)$ and $Y\sim K_{\theta,u,t}(\cdot\mid Z,H)$ are sampled at the **current online weights with gradients stopped**. The student minimizes $-\log K_{\theta,s,t}(Y\mid X_s,H)$. EMA weights are used only for validation selection and final inference. Existing CK-weight and interval warm-ups are retained.

A single CFM checkpoint is selected by the arithmetic mean of validation energy scores at **1 and 16 NFE**. All reported budgets share that checkpoint. Count-FM retains checkpoint selection at 64 tau steps, followed by validation-only sampler/budget selection. Test scores never determine selection.

The supplied configuration retains the full 30,000-update budget per candidate, two candidates per method, six sessions, and three seeds. Private diagnostic runs are separate from this experiment. Configurations marked `purpose="diagnostic"` do not export paper tables or figures. No diagnostic results are included in this notebook or package.

After all formal fits and evaluations finish, use `report` to rebuild tables and figures without training or generation. Incomplete experiments do not produce manuscript tables.


In [ ]:
output = run_experiment(CONFIG_PATH, root=ROOT, device=DEVICE, stage=STAGE)
print("Saved:", output)

## Main comparison

Equal session weights within each training seed; mean and sample SD across seeds. Timing is per forecast ensemble, including history encoding and count generation. All results, including unfavorable rankings, are retained.

In [ ]:
status_path = OUTPUT / "report_status.json"
if not status_path.exists() or json.loads(status_path.read_text())["status"] != "complete":
    raise RuntimeError("The formal experiment is incomplete. Resume training/evaluation before reporting.")
status = json.loads(status_path.read_text())
from countflow.neural_original import method_record
if any(status.get(k) != v for k, v in method_record(NEURAL_CONFIG).items()):
    raise RuntimeError("The saved report belongs to another method or a diagnostic run.")
summary = pd.read_csv(OUTPUT / "paper_neural_table.csv")
display(summary[["label", "energy_score_mean", "energy_score_std", "population_crps_mean", "population_crps_std", "neuron_rmse_mean", "neuron_rmse_std", "ensemble_ms_mean", "ensemble_ms_std"]])
print((OUTPUT / "paper_neural_table.tex").read_text())

In [ ]:
display(Image(filename=str(OUTPUT / "figures/paper_neural_main.png")))
print((OUTPUT / "figure_captions.txt").read_text())

## Appendix and audit results

The trace above forecasts each next bin using observed history. The appendix curves instead feed generated counts back into the model. Full-range panels retain all methods and extreme forecasts; linear detail panels use the same results. Both Count Flow Map budgets and matched Count-FM budgets are reported. The compact matched-budget table below averages sessions equally within seed, then reports the mean and sample SD across seeds. Per-run values remain in `matched_nfe_metrics.csv`.


In [ ]:
display(Image(filename=str(OUTPUT / "figures/paper_neural_appendix.png")))
display(pd.read_csv(OUTPUT / "paper_neural_sessions.csv"))
matched_summary = pd.read_csv(OUTPUT / "paper_neural_matched_nfe.csv")
display(matched_summary[["label", "energy_score_mean", "energy_score_std", "population_crps_mean", "population_crps_std", "neuron_rmse_mean", "neuron_rmse_std", "ensemble_ms_mean", "ensemble_ms_std"]])
display(pd.read_csv(OUTPUT / "dependence_control.csv"))
print("Vector PDFs and editable LaTeX tables:", OUTPUT)